# Experiment 1 · Notebook 07: Publish the `clf-v1.0.0` artifact

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Section 13.*

Writes the artifact the tutor service loads, `models/clf-v1.0.0/` (`pipeline.joblib`, `metadata.json`, `rules.yaml`), and
verifies it. **The artifact is a handoff, so its shape is treated like an API contract.** The metrics are final: nothing here
retrains differently, retunes or changes a number.

**What is shipped.** The pipeline **fitted on the TRAIN split only (860 rows)**: the exact object whose test metrics notebook 04
reports. It is *not* refitted on all 1,076 rows, because then the metrics in `metadata.json` would describe a different model
from the one being served.

**A published version is never overwritten.** If `models/clf-v1.0.0/` does not exist, this notebook runs in **build** mode and
writes it. If it exists, the notebook runs in **verify** mode: it writes nothing, loads the artifact and runs the same checks,
and any mismatch fails loudly. A re-run therefore cannot silently change `created_at` or the file bytes.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1-§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4-§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6-§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8-§9 | TF-IDF + LR, group-aware CV, the single test evaluation |
| `05_features_and_ood_probe` | §10-§11 | coefficient inspection, stability, length, OOD probe evaluation |
| `06_sensitivity_check` | §12 | robustness of the label mapping: Partial Correct dropped |
| `07_publish_artifact` | §13 | the `clf-v1.0.0` artifact: pipeline, metadata, rules, verification |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 07 to regenerate every file in `reports/experiments/`.

**Setup.** Imports, and this stage's inputs: the split rebuilt and **proved identical** by its fingerprint, and the final pipeline
fitted on train from the hyperparameters notebook 04 saved. A reproduction check asserts its test metrics equal notebook 04's
saved row to 4 decimals. The same cell computes every number that will go into `metadata.json` from the computed objects or the
committed CSVs (never retyped), and asserts each against its source.

In [1]:
import json, re, subprocess, sys, tempfile
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier

from confusion_baseline import MODELS_DIR, REPORTS_DIR, SEED
from confusion_baseline.artifact import sha256_file, validate_model_info
from confusion_baseline.data import DATASET_REPO, DATASET_REVISION, load_raw
from confusion_baseline.evaluate import metrics
from confusion_baseline.model import build_pipeline, length_baseline, load_params, token_count
from confusion_baseline.prep import LABEL_MAP_PRIMARY, clean, map_labels
from confusion_baseline.probe import load_probe, probe_sha256
from confusion_baseline.split import group_split, split_fingerprint

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 100)
FIG_DIR = REPORTS_DIR / "experiments"

# ---- inputs: the same split as notebooks 03-06 ----
df, followup_quarantine, load_report = load_raw()
del followup_quarantine
clean_df, _ = clean(df)
y_primary = map_labels(clean_df, "primary")
train, test = group_split(y_primary, test_size=0.2, seed=SEED)
U4_FINGERPRINT = "511fadfecbff1c84dad9f22c4ecdd51052b13c6c1187c6f60d2468342905e44c"
fp = split_fingerprint(test)
assert fp == U4_FINGERPRINT, fp
assert not set(train["group_key"]) & set(test["group_key"])
print(f"PASS: same test set as notebooks 03-06 (fingerprint {fp[:12]}...); train {len(train)} rows, test {len(test)} rows")

# ---- the pipeline to ship: fitted on TRAIN only ----
params = load_params(FIG_DIR / "chosen_params_primary.json")
final = build_pipeline(**params).fit(train["Answer"], train["y"])
assert list(final.classes_) == [0, 1]
print("PASS: pipeline.classes_ == [0, 1], so column 1 of predict_proba is P(confused)")

# ---- reproduction check against notebook 04's saved table (4 decimals) ----
pred_test = final.predict(test["Answer"])
y_te = test["y"].to_numpy()
cmp_ = pd.read_csv(FIG_DIR / "comparison_primary.csv").set_index("System")
col = {"accuracy": "Accuracy", "precision_confused": "Precision", "recall_confused": "Recall",
       "f1_confused": "F1 (confused)", "macro_f1": "Macro F1"}
m_model = metrics(y_te, pred_test)                                   # full precision
assert {col[k]: round(v, 4) for k, v in m_model.items()} == {c: round(float(cmp_.loc["TF-IDF + LR", c]), 4) for c in col.values()}
print(f"PASS: test metrics reproduce notebook 04's TF-IDF + LR row: F1(confused) {m_model['f1_confused']:.4f}, macro F1 {m_model['macro_f1']:.4f}")

# ---- baselines (full precision, asserted against the saved CSV) ----
X0_tr, X0_te = np.zeros((len(train), 1)), np.zeros((len(test), 1))
m_major = metrics(y_te, np.ones(len(test), dtype=int))                # majority class in train = confused
m_random = metrics(y_te, DummyClassifier(strategy="stratified", random_state=SEED).fit(X0_tr, train["y"]).predict(X0_te))
m_len = metrics(y_te, length_baseline(train).predict(token_count(test["Answer"])))
for name, mm in [("Majority class", m_major), ("Stratified random (seed 42)", m_random), ("Length-only LR", m_len)]:
    assert round(mm["f1_confused"], 4) == round(float(cmp_.loc[name, "F1 (confused)"]), 4), name
    assert round(mm["macro_f1"], 4) == round(float(cmp_.loc[name, "Macro F1"]), 4), name
print("PASS: baseline rows (majority, stratified seed 42, length-only) reproduce the saved comparison table")

# ---- OOD probe (evaluation only), asserted against notebook 05's saved table ----
probe = load_probe()
probe_pred = final.predict(probe["text"])
y_p = probe["y"].to_numpy()
probe_m = metrics(y_p, probe_pred)
probe_spec = float(((probe_pred == 0) & (y_p == 0)).sum() / (y_p == 0).sum())
probe_flagged = int(probe_pred.sum())
po = pd.read_csv(FIG_DIR / "probe_overall_primary.csv").set_index("System").loc["TF-IDF + LR"]
assert round(probe_m["accuracy"], 4) == round(float(po["Accuracy"]), 4)
assert round(probe_m["recall_confused"], 4) == round(float(po["Recall (confused)"]), 4)
assert round(probe_spec, 4) == round(float(po["Specificity"]), 4)
assert (len(probe), probe_flagged) == (40, 34), (len(probe), probe_flagged)
print(f"PASS: probe numbers reproduce notebook 05: accuracy {probe_m['accuracy']:.2f}, recall {probe_m['recall_confused']:.4f}, "
      f"specificity {probe_spec:.2f}, {probe_flagged} of {len(probe)} flagged confused")

# ---- sensitivity (committed CSV, notebook 06 row (a)) ----
sens = pd.read_csv(FIG_DIR / "comparison_drop_partial.csv").set_index("System").loc["TF-IDF + LR (a) primary config"]
print(f"PASS: loaded notebook 06 row (a): macro F1 {float(sens['Macro F1'])}, F1(confused) {float(sens['F1 (confused)'])}")
assert DATASET_REVISION == "e29cfdd4ec4179f54b2ea0409329207104de8bed"

PASS: same test set as notebooks 03-06 (fingerprint 511fadfecbff...); train 860 rows, test 216 rows
PASS: pipeline.classes_ == [0, 1], so column 1 of predict_proba is P(confused)
PASS: test metrics reproduce notebook 04's TF-IDF + LR row: F1(confused) 0.7581, macro F1 0.6629
PASS: baseline rows (majority, stratified seed 42, length-only) reproduce the saved comparison table
PASS: probe numbers reproduce notebook 05: accuracy 0.70, recall 0.8667, specificity 0.20, 34 of 40 flagged confused
PASS: loaded notebook 06 row (a): macro F1 0.6551, F1(confused) 0.6131


## 13. The `clf-v1.0.0` artifact

### 13.1 Which mode?

**Why.** Decision: never overwrite a published version. The directory's existence decides the mode, and the mode is printed.

In [2]:
ART_DIR = MODELS_DIR / "clf-v1.0.0"
MODE = "verify" if ART_DIR.exists() else "build"
print(f"artifact directory: models/{ART_DIR.name}/   exists: {ART_DIR.exists()}")
print(f"MODE = {MODE.upper()}", "(writes the artifact)" if MODE == "build" else "(writes NOTHING; loads and verifies the published artifact)")

artifact directory: models/clf-v1.0.0/   exists: True
MODE = VERIFY (writes NOTHING; loads and verifies the published artifact)


### 13.2 The rule layer (`rules.yaml`), authored here

**Why.** The probe (notebook 05) showed the model flags **8 of 10** explicit *understanding* messages as confused (specificity 0.20) and
answers "confused" to almost any short text, so the tutor needs explicit-phrase rules in **both** directions, versioned with the
model (ARCHITECTURE ADR-04). The header documents the matching rule exactly so the backend can implement it without guessing, and
lists the **known false triggers** instead of pretending there are none. The phrases are seeded from the spec and from the
explicit categories of the probe, with no-apostrophe variants (`dont`, `im`). The notebook renders the file from the lists below; in
build mode it writes it, and in verify mode it requires the file on disk to be identical to what is rendered here.

In [3]:
RULES_HEADER = [
    r"rules.yaml: explicit-phrase rule layer, versioned with the model (clf-v1.0.0). ADR-04.",
    r"",
    r"MATCHING RULE (implement exactly this; confusion_baseline/rules.py is the reference implementation):",
    r"  1. normalise the student message: lowercase; replace U+2018 and U+2019 with an ASCII apostrophe (');",
    r"     collapse every run of whitespace to one space; strip. Nothing else (punctuation is kept).",
    r"  2. match each phrase as a WHOLE-WORD SEQUENCE: regex  \b<phrase>\b  on the normalised text (re.escape the",
    r"     phrase). This is not a raw substring test: 'no idea' does not match 'no ideal'.",
    r"  3. check confusion_phrases first, then understanding_phrases, each in the order listed below.",
    r"     The FIRST match decides: confusion_phrases -> 'confused', understanding_phrases -> 'understood'.",
    r"  4. no match: fall through to the model (pipeline.predict_proba, column 1 = P(confused), threshold 0.5).",
    r"     A decision made by a phrase is reported with source = 'rule'; otherwise source = 'model'.",
    r"  Every phrase below is already in normalised form (lowercase, ASCII apostrophe, single spaces).",
    r"",
    r"KNOWN FALSE TRIGGERS AND LIMITS (not fixed; the phrase list is deliberately explicit):",
    r"  - 'got it' fires inside 'I got it from the docs': reported 'understood' (wrong). 'I got it wrong' is NOT a",
    r"    false trigger: 'got it wrong' is a confusion phrase, and the confusion list is checked first.",
    r"  - 'that makes sense' fires on 'that makes sense in theory but I cannot do it': reported 'understood'",
    r"    (the student is only partly following). 'none of that makes sense' is handled by checking the confusion",
    r"    list first (it has its own phrase), but novel negations are not.",
    r"  - 'doesn't make sense' / 'what do you mean' also occur when a student CHALLENGES the tutor's answer; they are",
    r"    treated as confusion on purpose, because that is the safer error for a tutor.",
    r"  - Phrases with extra words inserted do not match: 'i am so lost' misses (only 'i am lost' is listed) and falls",
    r"    through to the model. Whole-word matching prevents 'no ideal' / 'forgot it' style hits, not paraphrase.",
    r"  - 'i see now' fires on 'I see now that I was right': reported 'understood'.",
    r"  - Order matters: 'got it, but now I'm lost again' is 'confused' because the confusion list is checked first.",
    r"  - The phrase list is seeded from the spec and probe/ood_probe.yaml, so rule accuracy measured on that probe would",
    r"    be circular and is NOT reported anywhere.",
]
CONFUSION_PHRASES = [
    # spec seeds
    "i don't understand", "i dont understand", "i'm lost", "no idea", "what do you mean", "explain again",
    # variants
    "i do not understand", "i didn't understand", "i didnt understand", "i don't really understand", "i dont really understand",
    "im lost", "i am lost", "i'm completely lost", "im completely lost", "i'm totally lost", "im totally lost",
    "i'm confused", "im confused", "i am confused", "i'm so confused", "im so confused", "i'm still confused", "still confused",
    "don't get it", "dont get it", "don't follow", "dont follow", "i'm not following", "im not following",
    "not following how", "not following what", "no clue",
    # admitting a mistake or being stuck (checked before the understanding phrase 'got it')
    "got it wrong", "got that wrong", "idk", "i'm stuck", "im stuck", "i am stuck",
    "makes no sense", "doesn't make sense", "doesnt make sense", "does not make sense",
    "none of that makes sense", "none of this makes sense", "none of it makes sense", "nothing makes sense",
    "explain that again", "explain it again", "explain this again", "over my head",
]
UNDERSTANDING_PHRASES = [
    # spec seeds
    "that makes sense", "got it", "i see now",
    # variants
    "makes sense now", "now i see", "oh i see", "ok i see", "i get it now", "i understand now", "now i understand",
    "i understand it now", "that's clear", "thats clear", "that clears it up", "that cleared it up",
    "that's what i was missing", "thats what i was missing", "that's exactly what i was missing", "thats exactly what i was missing",
]
assert not set(CONFUSION_PHRASES) & set(UNDERSTANDING_PHRASES)
assert len(set(CONFUSION_PHRASES)) == len(CONFUSION_PHRASES) and len(set(UNDERSTANDING_PHRASES)) == len(UNDERSTANDING_PHRASES)

def render_rules_yaml():
    def block(name, phrases):
        return f"{name}:\n" + "".join(f"  - {json.dumps(p, ensure_ascii=False)}\n" for p in phrases)
    header = "".join(("# " + line).rstrip() + "\n" for line in RULES_HEADER)
    return header + "\n" + 'version: "1.0"\n\n' + block("confusion_phrases", CONFUSION_PHRASES) + "\n" + block("understanding_phrases", UNDERSTANDING_PHRASES)

RULES_TEXT = render_rules_yaml()
print(f"rules.yaml would have {len(RULES_TEXT.splitlines())} lines: {len(CONFUSION_PHRASES)} confusion phrases, {len(UNDERSTANDING_PHRASES)} understanding phrases")

rules.yaml would have 103 lines: 51 confusion phrases, 19 understanding phrases


### 13.3 Build the artifact (build mode only)

**Why.** In build mode this writes the three files, in a fixed order, from the objects computed in the setup cell. `metadata.json`
extends the spec's shape with the fields the service needs (split sizes and fingerprint, the input contract, the runtime, file
hashes) and carries a `model_info` block that matches the frontend's frozen `ModelInfo` contract exactly, so the service can return it
verbatim from `GET /model-info`.

**Input contract.** The pipeline takes the student's **raw text**. Notebook 02 section 4 showed that text normalisation changes no
TF-IDF token, and `TfidfVectorizer` lowercases internally, so the service must **not** preprocess. (The rule layer has its own
normalisation, described in `rules.yaml`.) In **verify** mode this cell writes nothing.

In [4]:
def to_jsonable(v):
    if isinstance(v, tuple):
        return list(v)
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    return str(v)

tf = final.named_steps["tfidf"]
lr = final.named_steps["clf"]
for step in final.named_steps.values():
    assert type(step).__module__.startswith("sklearn"), type(step)          # only sklearn objects inside
tfidf_keys = ["analyzer", "lowercase", "ngram_range", "min_df", "max_df", "max_features", "stop_words", "sublinear_tf",
              "use_idf", "smooth_idf", "norm", "token_pattern", "strip_accents", "binary"]
HYPER = {
    "tfidf": {**{k: to_jsonable(tf.get_params()[k]) for k in tfidf_keys}, "vocabulary_size": len(tf.vocabulary_)},
    "logistic_regression": {k: to_jsonable(v) for k, v in lr.get_params().items()},
    "selection": "GridSearchCV, 160 configs, StratifiedGroupKFold(5) on train, refit on macro F1",
}
METRIC_KEYS = ["accuracy", "precision_confused", "recall_confused", "f1_confused", "macro_f1"]
TEST_METRICS = {k: float(m_model[k]) for k in METRIC_KEYS}

def expected_metadata(created_at, files):
    # Everything here comes from computed objects or committed CSVs; created_at and files come from the disk / the build step.
    return {
        "version": "clf-v1.0.0",
        "created_at": created_at,
        "dataset": {"name": "JavaQuestionSynthFinal", "source": "huggingface.co/datasets/" + DATASET_REPO,
                    "revision": DATASET_REVISION, "rows_raw": int(load_report["rows_raw"]), "rows_after_cleaning": len(clean_df)},
        "label_map": dict(LABEL_MAP_PRIMARY),
        "split": {"strategy": "GroupShuffleSplit", "group_by": "Question", "test_size": 0.2, "seed": SEED,
                  "train_rows": len(train), "test_rows": len(test), "test_fingerprint": fp},
        "hyperparameters": HYPER,
        "metrics": dict(TEST_METRICS),
        "baselines": {"majority_f1_confused": float(m_major["f1_confused"]), "random_f1_confused": float(m_random["f1_confused"]),
                      "majority_macro_f1": float(m_major["macro_f1"]), "length_only_f1_confused": float(m_len["f1_confused"]),
                      "length_only_macro_f1": float(m_len["macro_f1"])},
        "ood_probe": {"accuracy": float(probe_m["accuracy"]), "recall_confused": float(probe_m["recall_confused"]),
                      "specificity_understood": probe_spec, "n_items": len(probe), "probe_sha256": probe_sha256(),
                      "note": f"diagnostic only; not used in training; {probe_flagged} of {len(probe)} flagged confused, "
                              "so the rule layer must handle both directions"},
        "sensitivity_drop_partial": {"macro_f1": float(sens["Macro F1"]), "f1_confused": float(sens["F1 (confused)"]),
                                     "note": "Partial Correct rows removed, same test questions, primary config refitted on 631 train rows "
                                             "(notebook 06, row (a)); macro F1 is close to the primary 0.6629, so the result is robust "
                                             "to the label mapping. Not comparable on F1(confused): the base rate differs."},
        "threshold": 0.5,
        "input": {"text": "raw student message, no preprocessing", "proba_index_confused": 1,
                  "classes": {"0": "understood", "1": "confused"}},
        "runtime": {"python": sys.version.split()[0], "scikit_learn": sklearn.__version__,
                    "note": "unpickle with this exact scikit-learn version"},
        "files": files,
        "caveat": "Labels are answer correctness used as a proxy for confusion, on a synthetic dataset. Not a measurement of "
                  "confusion. See experiment1Plan/02_DATASET.md and notebooks 05-06.",
        "model_info": {"version": "clf-v1.0.0", "trained_at": created_at,
                       "dataset": f"JavaQuestionSynthFinal@{DATASET_REVISION[:8]}", "threshold": 0.5, "metrics": dict(TEST_METRICS)},
    }

if MODE == "build":
    ART_DIR.mkdir(parents=True)
    joblib.dump(final, ART_DIR / "pipeline.joblib")
    (ART_DIR / "rules.yaml").write_text(RULES_TEXT, encoding="utf-8", newline="\n")
    created = datetime.now(timezone.utc).replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")
    files = {"pipeline.joblib": sha256_file(ART_DIR / "pipeline.joblib"), "rules.yaml": sha256_file(ART_DIR / "rules.yaml")}
    meta = expected_metadata(created, files)
    validate_model_info(meta["model_info"])
    (ART_DIR / "metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
    print("BUILD: wrote pipeline.joblib, rules.yaml, metadata.json to", ART_DIR.relative_to(ART_DIR.parents[1]).as_posix())
else:
    print("VERIFY: nothing written; the published artifact is checked below")
for name in ["pipeline.joblib", "rules.yaml", "metadata.json"]:
    p = ART_DIR / name
    print(f"  {name:16s} {p.stat().st_size:>8d} bytes   sha256 {sha256_file(p)}")

VERIFY: nothing written; the published artifact is checked below
  pipeline.joblib    283220 bytes   sha256 2c168f138aec44fcc19dc373c57818febd7276bb5bc56a0624290f9c65b47b51
  rules.yaml           4088 bytes   sha256 2f08da409552a03bdf632a8c67b7e5b20ca357345cdd9d1c045cf8ec6a7b3c9c
  metadata.json        3944 bytes   sha256 815b40dac55be0ba94cf02391ecc1858902f5ec08fe6297b4699193ad0dfdd67


### 13.4 Rule checks on new sentences

**Why.** Unit-style checks of the reference implementation (`confusion_baseline/rules.py`) on sentences written for this notebook, **none
taken from the probe**, including the opposite-meaning traps from the `rules.yaml` header. Cases marked *documented* assert the
behaviour the header already lists as a known limit, so a future change to the phrase list cannot alter it silently. **No rule accuracy is
reported on the probe**: the phrases were seeded from it, so that number would be circular.

In [5]:
from confusion_baseline.rules import load_rules, match

rules = load_rules(ART_DIR / "rules.yaml")
print("rules version:", rules["version"], "|", len(rules["confusion_phrases"]), "confusion,", len(rules["understanding_phrases"]), "understanding phrases")

CHECKS = [   # (sentence, expected label, note)
    ("ok that makes sense, thanks", "understood", ""),
    ("none of that makes sense to me", "confused", "trap: contains 'that makes sense'; the confusion list is checked first"),
    ("honestly this makes no sense", "confused", "'makes no sense' is its own phrase; 'makes sense' alone is not listed"),
    ("I got it wrong again", "confused", "'got it wrong' is a confusion phrase, checked before 'got it'"),
    ("I think I got that wrong", "confused", ""),
    ("I got it from the docs", "understood", "documented false trigger of 'got it'"),
    ("idk", "confused", ""),
    ("I'm stuck on the loop condition", "confused", ""),
    ("got it, but now I'm lost again", "confused", "order: confusion list first"),
    ("could you explain again how a stack works", "confused", ""),
    ("Im confused about recursion", "confused", "no-apostrophe variant"),
    ("I dont understand generics", "confused", "no-apostrophe spec seed"),
    ("what is a lambda expression", None, "no phrase: falls through to the model"),
    ("I\u2019m not sure which overload gets called", None, "hedging is the model's job, not a rule"),
    ("the loop prints 42, I see now why", "understood", ""),
    ("there is no ideal solution here", None, "whole-word guard: 'no idea' must not match 'no ideal'"),
    ("That doesn\u2019t make sense, 2 + 2 is 4", "confused", "documented: challenge to the tutor treated as confusion"),
    ("GOT   IT!", "understood", "normalisation: case, whitespace"),
    ("it makes sense now, thanks", "understood", ""),
    ("I'm completely lost on generics", "confused", ""),
    ("I am so lost", None, "documented limit: an inserted word defeats the phrase; falls through"),
    ("that makes sense in theory but how would that work in code", "understood", "documented false trigger (partial understanding)"),
]
rows = []
for text, want, note in CHECKS:
    got, phrase = match(text, rules)
    assert got == want, (text, got, want)
    rows.append({"sentence": text, "expected": want, "got": got, "matched phrase": phrase, "note": note})
display(pd.DataFrame(rows).fillna("none (model decides)"))
print(f"PASS: all {len(CHECKS)} rule checks behave as specified (including the documented false triggers)")
for label, key in [("confused", "confusion_phrases"), ("understood", "understanding_phrases")]:
    for p in rules[key]:
        got, hit = match(p, rules)
        assert got is not None, p          # every phrase fires on itself (an earlier phrase may win; checked next)
print("PASS: every listed phrase triggers some rule when it is the whole message")

rules version: 1.0 | 51 confusion, 19 understanding phrases


,sentence,expected,got,matched phrase,note
0,"ok that makes sense, thanks",understood,understood,that makes sense,
1,none of that makes sense to me,confused,confused,none of that makes sense,trap: contains 'that makes sense'; the confusion list is checked first
2,honestly this makes no sense,confused,confused,makes no sense,'makes no sense' is its own phrase; 'makes sense' alone is not listed
3,I got it wrong again,confused,confused,got it wrong,"'got it wrong' is a confusion phrase, checked before 'got it'"
4,I think I got that wrong,confused,confused,got that wrong,
5,I got it from the docs,understood,understood,got it,documented false trigger of 'got it'
6,idk,confused,confused,idk,
7,I'm stuck on the loop condition,confused,confused,i'm stuck,
8,"got it, but now I'm lost again",confused,confused,i'm lost,order: confusion list first
9,could you explain again how a stack works,confused,confused,explain again,


PASS: all 22 rule checks behave as specified (including the documented false triggers)
PASS: every listed phrase triggers some rule when it is the whole message


### 13.5 Fresh-process load check

**Why.** The service will unpickle `pipeline.joblib` without our package. This runs a separate Python process that imports only `joblib`,
`json` and `sys`, loads the pipeline, predicts on three strings, and asserts `confusion_baseline` was never imported. If the pickle
referenced our code, this fails.

In [6]:
script = '''
import json, sys
import joblib
path = sys.argv[1]
pipe = joblib.load(path)
assert [int(c) for c in pipe.classes_] == [0, 1]
samples = ["I dont understand how interfaces work", "that makes sense, thanks", "A constructor initialises a new object."]
proba = pipe.predict_proba(samples)[:, 1]
for s, p in zip(samples, proba):
    print(f"P(confused) = {p:.4f}   {s!r}")
assert "confusion_baseline" not in sys.modules, "the artifact must not depend on our package"
print("imports used:", sorted(m for m in ("joblib", "json", "sys", "sklearn", "numpy", "scipy") if m in sys.modules))
print("PASS: loaded and predicted with confusion_baseline NOT in sys.modules")
'''
with tempfile.TemporaryDirectory() as tmp:
    sp = f"{tmp}/fresh_load_check.py"
    open(sp, "w", encoding="utf-8").write(script)
    out = subprocess.run([sys.executable, sp, str(ART_DIR / "pipeline.joblib")], cwd=tmp, capture_output=True, text=True)
print(out.stdout, end="")
assert out.returncode == 0, out.stderr

P(confused) = 0.9408   'I dont understand how interfaces work'
P(confused) = 0.9918   'that makes sense, thanks'
P(confused) = 0.8619   'A constructor initialises a new object.'
imports used: ['joblib', 'json', 'numpy', 'scipy', 'sklearn', 'sys']
PASS: loaded and predicted with confusion_baseline NOT in sys.modules


### 13.6 Verification (both modes)

**Why.** Everything is re-read **from disk** and checked against the in-memory objects from the setup cell. In build mode this proves
what was just written; in verify mode it proves the published artifact still is what it claims, and any mismatch fails loudly.

In [7]:
FILES = {"pipeline.joblib", "rules.yaml", "metadata.json"}
assert {p.name for p in ART_DIR.iterdir()} == FILES, sorted(p.name for p in ART_DIR.iterdir())
print("PASS: the artifact directory contains exactly pipeline.joblib, metadata.json, rules.yaml")

disk = json.loads((ART_DIR / "metadata.json").read_text(encoding="utf-8"))
for name in ["pipeline.joblib", "rules.yaml"]:
    assert sha256_file(ART_DIR / name) == disk["files"][name], name
print("PASS: sha256 of pipeline.joblib and rules.yaml equal metadata['files']")

loaded = joblib.load(ART_DIR / "pipeline.joblib")
assert list(loaded.classes_) == [0, 1]
assert np.array_equal(loaded.predict(test["Answer"]), final.predict(test["Answer"]))
assert np.allclose(loaded.predict_proba(test["Answer"]), final.predict_proba(test["Answer"]), rtol=0, atol=0)
print("PASS: the loaded pipeline's predictions and probabilities on the test set equal the in-memory pipeline's")

for k, c in col.items():
    assert round(disk["metrics"][k], 4) == round(float(cmp_.loc["TF-IDF + LR", c]), 4), k
assert round(disk["baselines"]["majority_f1_confused"], 4) == round(float(cmp_.loc["Majority class", "F1 (confused)"]), 4)
assert round(disk["baselines"]["random_f1_confused"], 4) == round(float(cmp_.loc["Stratified random (seed 42)", "F1 (confused)"]), 4)
assert round(disk["baselines"]["majority_macro_f1"], 4) == round(float(cmp_.loc["Majority class", "Macro F1"]), 4)
assert round(disk["baselines"]["length_only_f1_confused"], 4) == round(float(cmp_.loc["Length-only LR", "F1 (confused)"]), 4)
assert round(disk["baselines"]["length_only_macro_f1"], 4) == round(float(cmp_.loc["Length-only LR", "Macro F1"]), 4)
print("PASS: metadata metrics and baselines equal comparison_primary.csv to 4 decimals")

assert re.fullmatch(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ", disk["created_at"]), disk["created_at"]
assert disk == expected_metadata(disk["created_at"], disk["files"]), "metadata.json differs from the values computed in this notebook"
print("PASS: every field of metadata.json equals the value computed here from the objects / committed CSVs")

validate_model_info(disk["model_info"])
assert disk["model_info"]["trained_at"] == disk["created_at"] and disk["model_info"]["metrics"] == disk["metrics"]
print("PASS: model_info has exactly {version, trained_at, dataset, threshold, metrics}; types match the frontend ModelInfo contract")

assert (ART_DIR / "rules.yaml").read_text(encoding="utf-8") == RULES_TEXT
print("PASS: rules.yaml on disk is identical to the rules rendered by this notebook")

assert disk["runtime"]["scikit_learn"] == sklearn.__version__, (disk["runtime"], sklearn.__version__)
assert disk["runtime"]["python"].split(".")[:2] == sys.version.split()[0].split(".")[:2]
assert disk["ood_probe"]["probe_sha256"] == probe_sha256() and disk["split"]["test_fingerprint"] == fp
print(f"PASS: runtime matches this environment (scikit-learn {sklearn.__version__}); probe sha256 and test fingerprint match")
print(f"\nALL CHECKS PASSED in {MODE.upper()} mode")

PASS: the artifact directory contains exactly pipeline.joblib, metadata.json, rules.yaml
PASS: sha256 of pipeline.joblib and rules.yaml equal metadata['files']
PASS: the loaded pipeline's predictions and probabilities on the test set equal the in-memory pipeline's
PASS: metadata metrics and baselines equal comparison_primary.csv to 4 decimals
PASS: every field of metadata.json equals the value computed here from the objects / committed CSVs
PASS: model_info has exactly {version, trained_at, dataset, threshold, metrics}; types match the frontend ModelInfo contract
PASS: rules.yaml on disk is identical to the rules rendered by this notebook
PASS: runtime matches this environment (scikit-learn 1.9.1); probe sha256 and test fingerprint match

ALL CHECKS PASSED in VERIFY mode


### 13.7 The artifact as published

**Why.** The final directory listing and the full `metadata.json`, as the service will read them.

In [8]:
print("models/clf-v1.0.0/")
for p in sorted(ART_DIR.iterdir()):
    print(f"  {p.name:18s} {p.stat().st_size:>8d} bytes")
print()
print((ART_DIR / "metadata.json").read_text(encoding="utf-8"))

models/clf-v1.0.0/
  metadata.json          3944 bytes
  pipeline.joblib      283220 bytes
  rules.yaml             4088 bytes

{
  "version": "clf-v1.0.0",
  "created_at": "2026-10-13T15:45:13Z",
  "dataset": {
    "name": "JavaQuestionSynthFinal",
    "source": "huggingface.co/datasets/nitinnovtech-ai-space/JavaQuestionSynthFinal",
    "revision": "e29cfdd4ec4179f54b2ea0409329207104de8bed",
    "rows_raw": 1078,
    "rows_after_cleaning": 1076
  },
  "label_map": {
    "Correct": "understood",
    "Partial Correct": "confused",
    "Wrong": "confused"
  },
  "split": {
    "strategy": "GroupShuffleSplit",
    "group_by": "Question",
    "test_size": 0.2,
    "seed": 42,
    "train_rows": 860,
    "test_rows": 216,
    "test_fingerprint": "511fadfecbff1c84dad9f22c4ecdd51052b13c6c1187c6f60d2468342905e44c"
  },
  "hyperparameters": {
    "tfidf": {
      "analyzer": "word",
      "lowercase": true,
      "ngram_range": [
        1,
        2
      ],
      "min_df": 1,
      "max_df": 1

## Findings of notebook 07

| Topic | Finding | Where |
|---|---|---|
| What is shipped | The pipeline **fitted on the 860-row train split only**, the object whose test metrics notebook 04 reports (accuracy 0.6898, F1(confused) 0.7581, macro F1 0.6629). Not a refit on all 1,076 rows, so the metadata describes the model that is served. | Setup, §13.3 |
| No overwrite | Build mode writes `models/clf-v1.0.0/` once; a re-run finds the directory and only verifies. | §13.1 |
| Contract | `metadata.json` extends the spec's shape (split sizes and fingerprint, input contract, runtime, file hashes) and carries a `model_info` block with exactly the frontend's five `ModelInfo` keys, so `GET /model-info` can return it verbatim. | §13.3, §13.6 |
| Input | Raw student text in, no preprocessing; `predict_proba` column 1 is P(confused); threshold 0.5. | §13.3 |
| Rule layer | `rules.yaml` holds explicit phrases in **both** directions, a documented matching rule and a documented list of false triggers. Reference implementation: `confusion_baseline/rules.py`. | §13.2, §13.4 |
| Rule checks | Checked on new sentences, including opposite-meaning traps; **no accuracy is reported on the probe**, because the phrases were seeded from it. | §13.4 |
| Portability | A fresh process that imports only `joblib` loads and runs the pipeline with `confusion_baseline` never imported. | §13.5 |
| Pinned runtime | The pickle must be loaded with the same scikit-learn version (recorded in the metadata and checked). | §13.6 |
| Caveat | The labels are a correctness proxy on a synthetic dataset; the OOD probe (34 of 40 flagged confused) shows the model alone is not a confusion detector. | metadata `caveat`, `ood_probe` |